In [0]:
%run ../functions/functions

In [0]:
%run ../functions/config

In [0]:
import pandas as pd
import os
from datetime import datetime
import pytz
from pyspark.sql import functions as f
 
storage_key = dbutils.secrets.get(scope="bbb", key="secret-stg-bbb")

In [0]:
fuso_br = pytz.timezone('America/Sao_Paulo')
agora = datetime.now(fuso_br)
data_hoje = agora.date()
particao_hoje = f"ano={agora.year}/mes={agora.month:02d}/dia={agora.day:02d}"

In [0]:
meu_storage = "stgbbb"
spark.conf.set(
    f"fs.azure.account.key.{meu_storage}.dfs.core.windows.net",
    storage_key
)

meu_container_destino = "bronze"
 
meu_container_origem = "landing"
caminho_origem = f"abfss://{meu_container_origem}@{meu_storage}.dfs.core.windows.net/"

In [0]:
conteudo_pasta = dbutils.fs.ls(caminho_origem)

In [0]:
for container in conteudo_pasta:

    caminho_destino = f"abfss://{meu_container_destino}@{meu_storage}.dfs.core.windows.net/{container.name}/{particao_hoje}/"

    conteudo_pasta_file = dbutils.fs.ls(f"{container.path}{particao_hoje}/")

    for arquivo_final in conteudo_pasta_file:       
        nome_arquivo_com_extensao = arquivo_final.name
        
        if(".csv" not in nome_arquivo_com_extensao):
            schema_selecionado = None
            nome_tabela = None
            for chave_identificadora, config in datasets.items():
                if config["prefixo_arquivo"] in nome_arquivo_com_extensao:
                    schema_selecionado=config["schema"]
                    nome_tabela = chave_identificadora
                    break

            if schema_selecionado is None:
                print(f"ALERTA: Arquivo {nome_arquivo_com_extensao} sem schema definido.")
                continue        


            nome_pasta_parquet = nome_arquivo_com_extensao.replace(".zip", "")
            files = dbutils.fs.ls(arquivo_final.path)
            single_file = files[0]
            path = unzip_files(single_file)

            origem = f"file:{path}"

            df_temp = spark.read.csv(origem, header=False, sep=';', encoding='latin1', schema=schema_selecionado)

            
            
        else:
            nome_pasta_parquet = nome_arquivo_com_extensao.replace(".csv", "")
        
            df_temp = spark.read.format("csv") \
                .option("header", "true") \
                .option("encoding", "latin1") \
                .option("sep", ";") \
                .option("multiLine", "true") \
                .load(arquivo_final.path)

        caminho_final = f"{caminho_destino}{nome_pasta_parquet}"

        df_final = df_temp.withColumn("dt_ingestao", f.lit(data_hoje))
            
        df_final.repartition("dt_ingestao").write.mode("overwrite").parquet(caminho_final)